In [ ]:
# bloco 1
import requests # permite conversar com ollama
import pandas as pd # cria e grava as tabelas de resultados

OLLAMA_URL = "http://localhost:11434" # endereço do ollama no pc
MODELOS = ["qwen2.5-coder:1.5b", "llama3.2:1b", "gemma3:1b"] # modelos que serão testados
TIMEOUT_S = 300                      # tempo máximo por resposta
ARQUIVO_RESULTADOS = "resultados_final.csv" # CSV onde gera os resultados, mudo o 
# ... nome para sempre começar uma nova tabela
ARQUIVO_TESTES = "teste_dos_resultados.csv" # bloco 8A grava aqui

In [ ]:
# bloco 2
# Exemplos próprios para demonstrar o fluxo. Substitua ou acrescente desafios conforme necessário.
DESAFIOS = [
    {
        "id": 1,
        "enunciado": """Dada uma lista de números inteiros, retorne a soma dos valores pares.
Exemplo: [1, 2, 3, 4] -> 6"""
    },
    {
        "id": 2,
        "enunciado": """Dada uma lista de números inteiros, conte quantos valores são positivos.
Exemplo: [-2, 0, 3, 5] -> 2"""
    },
]


In [ ]:
# bloco 3
# validar_desafios percorre o bloco 2 e monta uma lista de problemas
# se houver id repetido, anota que tem, se algum enunciado tiver vazio, anota qual ta vazio
def validar_desafios(desafios):
    erros = []
    ids = [d["id"] for d in desafios]
    if len(ids) != len(set(ids)):
        erros.append("há ids repetidos")
    for d in desafios:
        if not d["enunciado"].strip():
            erros.append(f"desafio {d['id']}: enunciado vazio")
    return erros

#se tiver algum problema, para o notebook e lista o que ta errado
erros = validar_desafios(DESAFIOS)
if erros:
    raise ValueError("Problemas nos desafios:\n- " + "\n- ".join(erros))

# mostra uma tabelinha (abaixo) com id e quantidade de caracteres
# serve apenas pra conferir visualmente mesmo, mostrar que nenhum veio vazio
# deixarei por estetica porém não faz diferença no código
pd.DataFrame([{"id": d["id"], "caracteres": len(d["enunciado"].strip())} for d in DESAFIOS])

In [ ]:
# bloco 4
# modelo do prompt, enunciado é onde entra os desafio
import re
TEMPLATE_PROMPT = """Resolva o desafio de programação abaixo em Java.
Leia da entrada padrão e escreva na saída padrão.
Enunciado:
{enunciado} 

Responda apenas com o código Java completo, sem explicações."""

# troca o enunciado pelo desafio e devolve o prompt pronto
def montar_prompt(enunciado):
    return TEMPLATE_PROMPT.format(enunciado=enunciado.strip())

# remove os marcadores ``` e devolve só o código, é usada só na hora de enviar como resposta pro desafio, pra não ter que ficar copiando e colando
# exemplo: em vez de o modelo devolver só print(1+1), ele devolve três crases + python na 
# primeira linha, depois print(1+1), e três crases na última linha; a função remove essas 
# duas linhas e deixa só o print(1+1).
def extrair_codigo(texto):
    
    m = re.search(r"```(?:\w+)?\n(.*?)```", texto, re.S)
    return (m.group(1) if m else texto).strip()    

# mostra o prompt do desafio 1 pra ver como vai ser mandado pras SLM
print(montar_prompt(DESAFIOS[0]["enunciado"]))

In [ ]:
# bloco 5
resp = requests.get(f"{OLLAMA_URL}/api/tags", timeout=10) # pergunta ollama os modelos instalados
resp.raise_for_status() # da erro se o ollama responder com falha

instalados = [m["name"] for m in resp.json()["models"]] 
for m in MODELOS:
    print(("OK    " if m in instalados else "FALTA ") + m)
# se um modelo faltar, a execução para aqui
faltando = [m for m in MODELOS if m not in instalados]
if faltando:
    raise RuntimeError(f"Modelos faltando: {faltando}. Use: ollama pull <modelo>")

In [ ]:
# bloco 6
import os
import time
from datetime import datetime, timezone, timedelta

# aqui é para as respostas serem sempre as mesmas, repetiveis
FUSO = timezone(timedelta(hours=-3))   # horário de Brasília
TEMPERATURA = 0
SEED = 42

# envia o prompt ao Ollama e devolve (output, tempo, status)
def consultar_modelo(modelo, prompt, timeout=None):
    timeout = timeout or TIMEOUT_S
    corpo = {
        "model": modelo,
        "prompt": prompt,
        "stream": False,
        "keep_alive": 0,   # descarrega o modelo após responder, esta linha é para não pesar meu computador
        "options": {"temperature": TEMPERATURA, "seed": SEED},
    }
    for tentativa in (1, 2):   # 1 tentativa nova só para falha de conexão
        inicio = time.time()
        try:
            r = requests.post(f"{OLLAMA_URL}/api/generate", json=corpo, timeout=timeout)
            tempo = round(time.time() - inicio, 2)
            if r.status_code != 200:
                return r.text, tempo, "erro"
            texto = r.json()["response"]
            return texto, tempo, ("ok" if texto.strip() else "vazio")

        # se passar do tempo, devolve timeout, se a conexão falhar tenta mais uma vez após 3 segundos
        #  e depois devolve erro, qualquer outro problema também vira erro
        except requests.exceptions.Timeout:
            return "", round(time.time() - inicio, 2), "timeout"
        except requests.exceptions.ConnectionError as e:
            if tentativa == 2:
                return str(e), round(time.time() - inicio, 2), "erro"
            time.sleep(3)
        except Exception as e:
            return str(e), round(time.time() - inicio, 2), "erro"

In [ ]:
# bloco 7 - função que grava no CSV, o bloco 9 usa ela pra registrar as respostas
# define a ordem das colunas do CSV
COLUNAS = ["id_desafio", "prompt", "modelo", "output",
           "tempo_resposta", "data_hora", "status"]

# monta uma linha com os dados da resposta e a data/hora de Brasília
def registrar(id_desafio, prompt, modelo, output, tempo, status, arquivo=None):
    arquivo = arquivo or ARQUIVO_RESULTADOS
    linha = {
        "id_desafio": id_desafio,
        "prompt": prompt,
        "modelo": modelo,
        "output": output,
        "tempo_resposta": tempo,
        "data_hora": datetime.now(FUSO).isoformat(timespec="seconds"),
        "status": status,
    }
    # escreve o cabeçalho só se o arquivo ainda não existe
    novo = not os.path.exists(arquivo) or os.path.getsize(arquivo) == 0

    # acrescenta a linha no fim do arquivo sem apagar as anteriores e devolve a linha.
    pd.DataFrame([linha], columns=COLUNAS).to_csv(
        arquivo, mode="a", header=novo, index=False, encoding="utf-8")
    return linha

In [ ]:
# bloco 8A  - pra testar
# pega o 1 desafio e monta um prompt
d = DESAFIOS[0]
prompt = montar_prompt(d["enunciado"])
for modelo in MODELOS:
    output, tempo, status = consultar_modelo(modelo, prompt)
    registrar(d["id"], prompt, modelo, output, tempo, status, arquivo=ARQUIVO_TESTES)
    print(f"{modelo} | status: {status} | tempo: {tempo}s")
    print(output)
    print("-" * 40)
# acima é apenas um teste rápido usando os modelos da lista, afinal o bloco 8A...
# ...é apenas um teste rápido, não é pra gerar o resultado final   

In [ ]:
# bloco 8B - pra testar
prompt = montar_prompt(DESAFIOS[0]["enunciado"])

testes = [
    # pede uma resposta a um modelo inexistente pra ver se volta erro
    ("modelo que não existe", "modelo-que-nao-existe", "oi", None, "erro"),
    # ve se volta timeout
    ("timeout de 1s", MODELOS[0], prompt, 1, "timeout"),
]
for nome, modelo, p, limite, esperado in testes:
    status = consultar_modelo(modelo, p, timeout=limite)[2]
    print(("OK " if status == esperado else "FALHOU ") + nome + f" (veio: {status})")

In [ ]:
# bloco 9
# lê o CSV e lista o que já foi gravado com status ok (desafio + modelo + prompt)
def combinacoes_ok():
    """Combinações (id, modelo, prompt) já gravadas com status ok."""
    if not os.path.exists(ARQUIVO_RESULTADOS) or os.path.getsize(ARQUIVO_RESULTADOS) == 0:
        return set()
    df = pd.read_csv(ARQUIVO_RESULTADOS)
    df = df[df["status"] == "ok"]
    return set(zip(df["id_desafio"].astype(int), df["modelo"], df["prompt"]))

# percorre cada desafio com cada um dos 3 modelos, pula o que já está ok, e no resto 
# consulta o modelo e grava a linha
def executar_ciclo():
    feitas = combinacoes_ok()
    total = len(DESAFIOS) * len(MODELOS)
    n = pulados = falhas = 0
    for d in DESAFIOS:
        prompt = montar_prompt(d["enunciado"])
        for modelo in MODELOS:
            n += 1
            if (int(d["id"]), modelo, prompt) in feitas:
                print(f"[{n}/{total}] pulando desafio {d['id']} + {modelo} (já está ok)")
                pulados += 1
                continue
            print(f"[{n}/{total}] desafio {d['id']} + {modelo} ...", end=" ", flush=True)
            output, tempo, status = consultar_modelo(modelo, prompt)
            registrar(d["id"], prompt, modelo, output, tempo, status)
            if status != "ok":
                falhas += 1
            print(f"{status} ({tempo}s)")
    print(f"Fim: {pulados} puladas | {total - pulados - falhas} novas ok | {falhas} com falha")

executar_ciclo()

In [ ]:
# bloco 10
# mostra a tabela final (só a última linha de cada desafio + modelo) e a contagem por status
if not os.path.exists(ARQUIVO_RESULTADOS):
    # se o bloco 9 ainda n rodou:
    raise FileNotFoundError("Rode o bloco 9 primeiro: ainda não há resultados.")

resultados = pd.read_csv(ARQUIVO_RESULTADOS)
# so a tentativa mais recente de cada combinação:
final = resultados.drop_duplicates(["id_desafio", "modelo"], keep="last")

print(final["status"].value_counts().to_string())
final[["id_desafio", "modelo", "status", "tempo_resposta", "data_hora"]]

In [ ]:
# bloco 11 - configuração da coleta (usa os blocos 1 a 7)
import json
from pathlib import Path
from urllib import robotparser
from urllib.parse import urlparse

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

DEBUG_ADDR = "127.0.0.1:9222"        # Chrome aberto com a porta de controle ( não irei usar conta pessoal para pegar os desafios, por isso abrir outro chrome)
PASTA_CACHE = Path("cache")          # um arquivo .json por desafio
PAUSA_S = 5                          # pausa entre páginas, em segundos
IDENTIFICACAO = "projeto-academico-slm/1.0"

# desafios:
URLS = [
    {"id": 1, "url": "https://leetcode.com/problems/two-sum/"},
    {"id": 2, "url": "https://leetcode.com/problems/palindrome-number/"},
    {"id": 3, "url": "https://leetcode.com/problems/roman-to-integer/"},
    {"id": 4, "url": "https://leetcode.com/problems/valid-parentheses/"},
    {"id": 5, "url": "https://leetcode.com/problems/merge-two-sorted-lists/"},
    {"id": 6, "url": "https://leetcode.com/problems/best-time-to-buy-and-sell-stock/"},
    {"id": 7, "url": "https://leetcode.com/problems/valid-palindrome/"},
    {"id": 8, "url": "https://leetcode.com/problems/climbing-stairs/"},
    {"id": 9, "url": "https://leetcode.com/problems/contains-duplicate/"},
    {"id": 10, "url": "https://leetcode.com/problems/maximum-subarray/"},
]

_ROBOTS = {}

def permitido_pelo_robots(url):
    #True só se o robots.txt do site permitir esta URL
    site = "{0.scheme}://{0.netloc}".format(urlparse(url))
    if site not in _ROBOTS:
        try:
            r = requests.get(site + "/robots.txt", timeout=10,
                             headers={"User-Agent": IDENTIFICACAO})
            if r.status_code == 200:
                rp = robotparser.RobotFileParser()
                rp.parse(r.text.splitlines())
                _ROBOTS[site] = rp
            else:
                _ROBOTS[site] = None
        except requests.exceptions.RequestException:
            _ROBOTS[site] = None
    rp = _ROBOTS[site]
    return rp is not None and rp.can_fetch(IDENTIFICACAO, url)

In [ ]:
# bloco 12 - conecta ao Chrome já logado e lê o enunciado
SELETORES_ENUNCIADO = [
    '[data-track-load="description_content"]',
    'div[class*="description"]',
]

def conectar_chrome():
    #Conecta ao Chrome já aberto (não abre nem fecha nada).
    opcoes = Options()
    opcoes.add_experimental_option("debuggerAddress", DEBUG_ADDR)
    return webdriver.Chrome(options=opcoes)

def ler_pagina(driver):
    titulo = driver.title
    for seletor in SELETORES_ENUNCIADO:
        achados = driver.find_elements(By.CSS_SELECTOR, seletor)
        if achados and achados[0].text.strip():
            return titulo, achados[0].text.strip()
    return titulo, ""

def esperar_enunciado(driver, limite=20):
    fim = time.time() + limite
    while time.time() < fim:
        titulo, texto = ler_pagina(driver)
        if texto:
            return titulo, texto
        time.sleep(1)
    return ler_pagina(driver)

In [ ]:
# bloco 13 - verificação automática, cache e coleta de um desafio
SINAIS_DE_BLOQUEIO = ["sign in to", "log in to", "captcha",
                      "verify you are human", "subscribe to unlock", "premium"]

def checar(dados):
    #Lista de problemas encontrados (vazia = passou).
    problemas = []
    if not dados["texto"].strip():
        problemas.append("enunciado vazio")
    if "accounts" in dados["url_final"].lower():
        problemas.append("redirecionou para página de login")
    inicio = (dados["titulo"] + " " + dados["texto"][:500]).lower()
    if any(s in inicio for s in SINAIS_DE_BLOQUEIO):
        problemas.append("página de login, captcha ou conteúdo premium")
    partes = urlparse(dados["url"]).path.strip("/").split("/")
    slug = partes[1] if len(partes) > 1 else ""
    if slug and not all(p in dados["titulo"].lower() for p in slug.split("-")):
        problemas.append("título não bate com a URL")
    return problemas

def mostrar_verificacao(dados, problemas):
    print(f"id: {dados['id']}")
    print(f"URL: {dados['url']}")
    print(f"título: {dados['titulo']}")
    print(f"tamanho do texto: {len(dados['texto'])} caracteres")
    print("--- início do enunciado ---")
    print(dados["texto"][:400])
    print("---------------------------")
    print("checagem automática:", "OK" if not problemas else "FALHOU: " + "; ".join(problemas))

def coletar(item, driver):
    arquivo = PASTA_CACHE / f"{item['id']}.json"
    if arquivo.exists():   # cache: não acessa o site de novo
        return json.loads(arquivo.read_text(encoding="utf-8"))
    if not permitido_pelo_robots(item["url"]):
        raise PermissionError(f"robots.txt não permite acessar {item['url']}")
    driver.get(item["url"])
    titulo, texto = esperar_enunciado(driver)
    dados = {"id": item["id"], "url": item["url"], "url_final": driver.current_url,
             "titulo": titulo, "texto": texto,
             "coletado_em": datetime.now(FUSO).isoformat(timespec="seconds")}
    if not checar(dados):   # só guarda no cache se passou na checagem
        PASTA_CACHE.mkdir(exist_ok=True)
        arquivo.write_text(json.dumps(dados, ensure_ascii=False, indent=2), encoding="utf-8")
    time.sleep(PAUSA_S)
    return dados

In [ ]:
# bloco 14 - teste mínimo: coleta só a URL 1 e mostra a verificação (não envia nada aos modelos)
# só pra verificar se está tudo ok
driver = conectar_chrome()
dados = coletar(URLS[0], driver)
mostrar_verificacao(dados, checar(dados))
# usa dados falsos que imitam uma página de login (texto vazio, endereço de login, título “Sign in”) para ver se a checagem os pega.

In [ ]:
#bloco 14B
falso = {"id": 0, "url": "https://leetcode.com/problems/two-sum/",
         "url_final": "https://leetcode.com/accounts/login/",
         "titulo": "Sign in", "texto": ""}
print(checar(falso))   # esperado: lista com 3 ou 4 problemas
# serve para ver se o bot ta acessando tudo corretamente, se ele pega o enunciado, cai na pagina de login, se o titulo nao bate com a URLsi

In [ ]:
# bloco 15 - coleta as 10 URLs, mostra a verificação e pede sua confirmação
CONFIRMADOS = {}   # id -> dados que você confirmou (some se reiniciar o notebook)
SINAIS_DE_PARADA = ("redirecionou", "login, captcha")

def coletar_todos():
    driver = conectar_chrome()   # não fecha o seu Chrome
    for item in URLS:
        if item["id"] in CONFIRMADOS:
            continue
        try:
            dados = coletar(item, driver)
        except PermissionError as e:
            print("PULADO:", e)
            continue
        problemas = checar(dados)
        mostrar_verificacao(dados, problemas)
        if problemas:
            print("A checagem falhou: este desafio NÃO segue para os modelos.")
            if any(s in p for p in problemas for s in SINAIS_DE_PARADA):
                print("Possível login ou captcha: parei aqui. Resolva no Chrome e rode de novo.")
                break
            continue
        resposta = ""
        while resposta not in ("sim", "não", "nao", "pular"):
            resposta = input("Este é o desafio correto? (sim / não / pular): ").strip().lower()
        if resposta == "sim":
            CONFIRMADOS[item["id"]] = dados
        else:
            print("Não confirmado: não será enviado aos modelos.")
    print(f"Confirmados: {sorted(CONFIRMADOS)}")

coletar_todos()

In [ ]:
# bloco 16 - prompt do LeetCode e arquivo de resultados novo (usa os blocos 1 a 7)
ARQUIVO_LEETCODE = "resultados_leetcode.csv"
ARQUIVO_TESTES_LEETCODE = "teste_leetcode.csv"
COLUNAS_LEETCODE = COLUNAS + ["url_desafio"]

TEMPLATE_LEETCODE = """Resolva o desafio de programação abaixo em Java.
Escreva a classe Solution com o método público que resolve o problema, no formato do LeetCode. Inclua os imports necessários.

Enunciado:
{enunciado}

Responda apenas com o código Java completo, sem explicações."""

def montar_prompt_leetcode(enunciado):
    return TEMPLATE_LEETCODE.format(enunciado=enunciado.strip())

def registrar_leetcode(id_desafio, prompt, modelo, output, tempo, status, url, arquivo=None):
    arquivo = arquivo or ARQUIVO_LEETCODE
    linha = {
        "id_desafio": id_desafio,
        "prompt": prompt,
        "modelo": modelo,
        "output": output,
        "tempo_resposta": tempo,
        "data_hora": datetime.now(FUSO).isoformat(timespec="seconds"),
        "status": status,
        "url_desafio": url,
    }
    novo = not os.path.exists(arquivo) or os.path.getsize(arquivo) == 0
    pd.DataFrame([linha], columns=COLUNAS_LEETCODE).to_csv(
        arquivo, mode="a", header=novo, index=False, encoding="utf-8")
    return linha

# mostra o prompt do primeiro desafio confirmado, para conferir
if not CONFIRMADOS:
    raise RuntimeError("Nenhum desafio confirmado: rode o bloco 15 primeiro.")
primeiro = sorted(CONFIRMADOS)[0]
print(montar_prompt_leetcode(CONFIRMADOS[primeiro]["texto"]))

In [ ]:
# bloco 17 - teste mínimo: desafio confirmado mais baixo × primeiro modelo da lista
dados = CONFIRMADOS[primeiro]
prompt = montar_prompt_leetcode(dados["texto"])
modelo = MODELOS[0]

output, tempo, status = consultar_modelo(modelo, prompt)
registrar_leetcode(dados["id"], prompt, modelo, output, tempo, status,
                   dados["url"], arquivo=ARQUIVO_TESTES_LEETCODE)
print(f"{modelo} | status: {status} | tempo: {tempo}s")
print(output)

In [ ]:
# bloco 18 - 10 desafios × 3 modelos, grava em resultados_leetcode.csv
def combinacoes_ok_leetcode():
    if not os.path.exists(ARQUIVO_LEETCODE) or os.path.getsize(ARQUIVO_LEETCODE) == 0:
        return set()
    df = pd.read_csv(ARQUIVO_LEETCODE)
    df = df[df["status"] == "ok"]
    return set(zip(df["id_desafio"].astype(int), df["modelo"], df["prompt"]))

def executar_ciclo_leetcode():
    if not CONFIRMADOS:
        raise RuntimeError("Nenhum desafio confirmado: rode o bloco 15 primeiro.")
    feitas = combinacoes_ok_leetcode()
    ids = sorted(CONFIRMADOS)
    total = len(ids) * len(MODELOS)
    n = pulados = falhas = 0
    for id_desafio in ids:
        dados = CONFIRMADOS[id_desafio]
        prompt = montar_prompt_leetcode(dados["texto"])
        for modelo in MODELOS:
            n += 1
            if (int(id_desafio), modelo, prompt) in feitas:
                print(f"[{n}/{total}] pulando desafio {id_desafio} + {modelo} (já está ok)")
                pulados += 1
                continue
            print(f"[{n}/{total}] desafio {id_desafio} + {modelo} ...", end=" ", flush=True)
            output, tempo, status = consultar_modelo(modelo, prompt)
            registrar_leetcode(id_desafio, prompt, modelo, output, tempo, status, dados["url"])
            if status != "ok":
                falhas += 1
            print(f"{status} ({tempo}s)")
    print(f"Fim: {pulados} puladas | {total - pulados - falhas} novas ok | {falhas} com falha")

executar_ciclo_leetcode()

In [ ]:
# bloco 19 - última tentativa de cada desafio + modelo, e contagem por status
if not os.path.exists(ARQUIVO_LEETCODE):
    raise FileNotFoundError("Rode o bloco 18 primeiro: ainda não há resultados.")
resultados_lc = pd.read_csv(ARQUIVO_LEETCODE)
final_lc = resultados_lc.drop_duplicates(["id_desafio", "modelo"], keep="last")

print(final_lc["status"].value_counts().to_string())
final_lc[["id_desafio", "modelo", "status", "tempo_resposta", "data_hora", "url_desafio"]]

 O que faz cada célula

1. **Configuração:** URL do Ollama, modelos, tempo limite e nomes dos arquivos
2. **Desafios:** lista com id e enunciado de cada desafio
3. **Validação:** confere ids repetidos e enunciados vazios
4. **Prompt:** monta o prompt de cada desafio (e a função que tira as crases do código)
5. **Checagem do Ollama:** confere se o Ollama e os 3 modelos estão disponíveis
6. **Consulta ao modelo:** envia o prompt, mede o tempo e devolve o status
7. **Registro:** grava uma linha no CSV
8. **8A / 8B (testes):** 8A testa os 3 modelos e grava só no arquivo de teste; 8B simula erro e timeout, sem gravar.
9. **Ciclo completo:** repete para todos os desafios e modelos, pulando o que já está ok
10. **Resultados:** mostra a tabela final e a contagem por status
11. **Configuração da coleta:** lista das URLs dos desafios do LeetCode, pasta do cache, pausa entre páginas e a checagem do robots.txt do site
12. **Chrome e leitura da página:** conecta ao Chrome já aberto e logado (sem abrir nem fechar nada) e lê o título e o enunciado da página
13. **Verificação e cache:** confere se o enunciado não está vazio, se não é página de login, captcha ou página do premium e se o título bate com a URL; guarda o desafio em `cache/` só se passar e sem acessar o site de novo
14. **14 / 14B (testes da coleta):** 14 coleta só o primeiro desafio e mostra a verificação; 14B simula uma página de login para provar que a checagem acusa problemas, sem acessar o site
15. **Coleta dos 10 desafios:** percorre as URLs, mostra a verificação de cada uma e pede a confirmação (sim / não / pular), só os confirmados seguem, aqui é para se aparecer login ou captcha
16. **Prompt e registro do LeetCode:** monta o prompt e define a gravação no CSV novo, que tem a coluna `url_desafio` além das 7 colunas de antes
17. **Teste com o modelo:** envia 1 desafio confirmado a 1 modelo e grava só no arquivo de teste (`teste_leetcode.csv`)
18. **Ciclo completo do LeetCode:** repete para os desafios confirmados × os 3 modelos, pulando o que já está `ok`, e grava em `resultados_leetcode.csv`
19. **Resultados do LeetCode:** mostra a última tentativa de cada desafio + modelo e a contagem por status

### Preparação dos arquivos públicos
A célula seguinte gera `desafios.csv` e cópias dos CSVs sem a coluna `prompt`. Os arquivos locais originais permanecem intactos.

In [2]:
# bloco 20 — preparar arquivos para publicação
import hashlib
import json
import re
import pandas as pd
from pathlib import Path

# Define os caminhos independentemente da execução de outros blocos
PASTA_CACHE = Path("cache")
PASTA_CACHE.mkdir(exist_ok=True)

def normalizar_enunciado(texto):
    return " ".join(str(texto or "").split())

def identificar_enunciado(texto_prompt):
    encontrado = re.search(
        r"Enunciado:\s*(.*?)\s*(?:Responda apenas|$)",
        str(texto_prompt),
        flags=re.IGNORECASE | re.DOTALL
    )
    texto = encontrado.group(1) if encontrado else texto_prompt
    return normalizar_enunciado(texto)

def gerar_hash_enunciado(texto):
    texto = normalizar_enunciado(texto)
    return hashlib.sha256(texto.encode("utf-8")).hexdigest()

# 1. Índice dos desafios coletados localmente
linhas_desafios = []
hashes_adicionados = set()

for arquivo_json in sorted(PASTA_CACHE.glob("*.json")):
    dados = json.loads(arquivo_json.read_text(encoding="utf-8"))
    enunciado = normalizar_enunciado(dados.get("texto", ""))
    if not enunciado:
        continue
    hash_id = gerar_hash_enunciado(enunciado)
    hashes_adicionados.add(hash_id)
    titulo = re.sub(r"\s*-\s*LeetCode\s*$", "", dados.get("titulo", ""), flags=re.I).strip()
    linhas_desafios.append({
        "id_desafio": f"leetcode-{dados.get('id')}",
        "titulo": titulo,
        "url": dados.get("url", ""),
        "caracteres": len(enunciado),
        "hash_enunciado": hash_id,
        "coletado_em": dados.get("coletado_em", ""),
        "fonte": "LeetCode"
    })

# 2. Inclui referências aos enunciados presentes nos CSVs locais sem publicar o texto
titulos_conhecidos = [
    ("soma apenas dos números pares", "Exercício de soma dos números pares", "Exercício próprio"),
    ("notas referentes às duas avaliações", "Média de notas (registro histórico BeeCrowd)", "BeeCrowd — registro histórico"),
    ("retorne true se ela for um palíndromo", "Verificação de palíndromo", "Exercício de teste")
]
historicos = {}
arquivos_origem = [
    "resultados_antigo_21do10.csv",
    "resultados_final.csv",
    "resultados_leetcode.csv",
    "teste_dos_resultados.csv",
    "teste_leetcode.csv"
]
for nome_arquivo in arquivos_origem:
    caminho = Path(nome_arquivo)
    if not caminho.exists():
        continue
    tabela = pd.read_csv(caminho)
    if "prompt" not in tabela.columns:
        continue
    for _, linha in tabela.drop_duplicates(subset=["id_desafio", "prompt"]).iterrows():
        enunciado = identificar_enunciado(linha["prompt"])
        if not enunciado:
            continue
        hash_id = gerar_hash_enunciado(enunciado)
        if hash_id in hashes_adicionados or hash_id in historicos:
            continue
        titulo = f"Exercício histórico {linha['id_desafio']}"
        fonte = "Registro histórico"
        for trecho, nome, origem in titulos_conhecidos:
            if trecho in enunciado.lower():
                titulo, fonte = nome, origem
                break
        historicos[hash_id] = {
            "id_desafio": f"historico-{len(historicos) + 1}",
            "titulo": titulo,
            "url": "",
            "caracteres": len(enunciado),
            "hash_enunciado": hash_id,
            "coletado_em": "",
            "fonte": fonte
        }

linhas_desafios.extend(historicos.values())
pd.DataFrame(linhas_desafios).to_csv(
    "desafios.csv", index=False, encoding="utf-8-sig"
)

# 3. Cria cópias públicas dos CSVs, trocando prompt pelo hash do enunciado
pasta_publica = Path("resultados")
pasta_publica.mkdir(exist_ok=True)

for nome_arquivo in arquivos_origem:
    caminho = Path(nome_arquivo)
    if not caminho.exists():
        continue
    tabela = pd.read_csv(caminho)
    if "prompt" in tabela.columns:
        posicao = tabela.columns.get_loc("prompt")
        hashes = tabela["prompt"].map(
            lambda prompt: gerar_hash_enunciado(identificar_enunciado(prompt))
        )
        tabela = tabela.drop(columns=["prompt"])
        tabela.insert(posicao, "hash_enunciado", hashes)
    tabela.to_csv(
        pasta_publica / caminho.name,
        index=False,
        encoding="utf-8-sig"
    )

print("Arquivos públicos preparados: desafios.csv e resultados/*.csv")
print("Os CSVs originais e a pasta cache/ não foram alterados.")


Arquivos públicos preparados: desafios.csv e resultados/*.csv
Os CSVs originais e a pasta cache/ não foram alterados.
